# Daily Scanner + Stockbee

**Run:** Pre-market or after market close

**Scans:** Bearish Momentum · VCP · Satish · Hardik · NR6 · Rocket Base · EP Breakout · 20d/2m High · Gap Up · Up 4%+ · Stockbee (5 scans)

**Universes:** S&P 500 · Russell 2000 · Nifty 500 (toggle with `INCLUDE_NIFTY`)

**Publishes to:** https://docs.google.com/spreadsheets/d/1rzc_6fZoHMFi1Ee75zRmIuGxeCWog1E62pZp9c1Lsfs

**Runtime > Run all**

In [ ]:
# CELL 1 — Install
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "yfinance", "tqdm", "requests", "lxml", "gspread", "gspread-dataframe"])
print("Ready")

In [ ]:
# CELL 2 — Settings
import warnings; warnings.filterwarnings("ignore")
import requests, io, time, gc, os
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime

# ── Universe / market mode ──────────────────────────────────
SCAN_MARKET = os.environ.get("SCAN_MARKET", "both").strip().lower()
if SCAN_MARKET not in {"india", "usa", "both"}:
    raise ValueError(f"SCAN_MARKET must be 'india', 'usa', or 'both', got: {SCAN_MARKET!r}")

if SCAN_MARKET == "india":
    INCLUDE_NIFTY   = True
    INCLUDE_SP500   = False
    INCLUDE_RUSSELL = False
elif SCAN_MARKET == "usa":
    INCLUDE_NIFTY   = False
    INCLUDE_SP500   = True
    INCLUDE_RUSSELL = False   # set True if you want Russell too
else:  # both
    INCLUDE_NIFTY   = True
    INCLUDE_SP500   = True
    INCLUDE_RUSSELL = False   # keep off by default — slow

print(f"SCAN_MARKET : {SCAN_MARKET.upper()}")
print(f"  S&P 500   : {INCLUDE_SP500}")
print(f"  Russell   : {INCLUDE_RUSSELL}")
print(f"  Nifty 500 : {INCLUDE_NIFTY}")
BATCH_SIZE            = 40
SLEEP_BETWEEN_BATCHES = 3
SHEET_ID = "1rzc_6fZoHMFi1Ee75zRmIuGxeCWog1E62pZp9c1Lsfs"
print(f"Nifty included: {INCLUDE_NIFTY}")
print(f"Sheet: https://docs.google.com/spreadsheets/d/{SHEET_ID}")

In [ ]:
# CELL 3 — Load tickers
HEADERS = {"User-Agent":"Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"}

def get_sp500():
    try:
        html = requests.get("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",headers=HEADERS,timeout=15).text
        t = [str(x).replace(".","-") for x in pd.read_html(io.StringIO(html))[0]["Symbol"].tolist()]
        print(f"  S&P 500      : {len(t)}"); return t
    except Exception as e:
        print(f"  SP500 failed: {e}"); return []

def get_russell2000():
    try:
        nyse = requests.get("https://raw.githubusercontent.com/rreichel3/US-Stock-Symbols/main/nyse/nyse_tickers.txt",headers=HEADERS,timeout=15).text.strip().split()
        nasd = requests.get("https://raw.githubusercontent.com/rreichel3/US-Stock-Symbols/main/nasdaq/nasdaq_tickers.txt",headers=HEADERS,timeout=15).text.strip().split()
        sp_csv = requests.get("https://raw.githubusercontent.com/datasets/s-and-p-500-companies/main/data/constituents.csv",headers=HEADERS,timeout=15).text
        sp_set = set(pd.read_csv(io.StringIO(sp_csv))["Symbol"].str.replace(".","-",regex=False).tolist())
        all_t = list(dict.fromkeys(nyse+nasd))
        t = [x for x in all_t if x.isalpha() and 2<=len(x)<=4 and x not in sp_set][:2000]
        print(f"  Russell 2000 : {len(t)}"); return t
    except Exception as e:
        print(f"  R2K failed: {e}"); return []

def get_nifty500():
    symbols = [
        "360ONE","3MINDIA","ABB","ACC","ACMESOLAR","AIAENG","APLAPOLLO","AUBANK",
        "AWL","AADHARHFC","AARTIIND","AAVAS","ABBOTINDIA","ACE","ACUTAAS","ADANIENSOL",
        "ADANIENT","ADANIGREEN","ADANIPORTS","ADANIPOWER","ATGL","ABCAPITAL","ABFRL","ABLBL",
        "ABREL","ABSLAMC","CPPLUS","AEGISLOG","AEGISVOPAK","AFCONS","AFFLE","AJANTPHARM",
        "ALKEM","ABDL","AMBER","AMBUJACEM","ANANDRATHI","ANANTRAJ","ANGELONE","ANTHEM",
        "ANURAS","APARINDS","APOLLOHOSP","APOLLOTYRE","APTUS","ASAHIINDIA","ASHOKLEY","ASIANPAINT",
        "ASTERDM","ASTRAL","ATHERENERG","ATUL","AUROPHARMA","AIIL","DMART","AXISBANK",
        "BEML","BLS","BSE","BAJAJ-AUTO","BAJFINANCE","BAJAJFINSV","BAJAJHLDNG","BAJAJHFL",
        "BALKRISIND","BALRAMCHIN","BANDHANBNK","BANKBARODA","BANKINDIA","MAHABANK","BATAINDIA","BAYERCROP",
        "BELRISE","BERGEPAINT","BDL","BEL","BHARATFORG","BHEL","BPCL","BHARTIARTL",
        "BHARTIHEXA","BIKAJI","GROWW","BIOCON","BSOFT","BLUEDART","BLUEJET","BLUESTARCO",
        "BBTC","BOSCHLTD","FIRSTCRY","BRIGADE","BRITANNIA","MAPMYINDIA","CCL","CESC",
        "CGPOWER","CIEINDIA","CRISIL","CANFINHOME","CANBK","CANHLIFE","CAPLIPOINT","CGCL",
        "CARBORUNIV","CARTRADE","CASTROLIND","CEATLTD","CEMPRO","CENTRALBK","CDSL","CHALET",
        "CHAMBLFERT","CHENNPETRO","CHOICEIN","CHOLAHLDNG","CHOLAFIN","CIPLA","CUB","CLEAN",
        "COALINDIA","COCHINSHIP","COFORGE","COHANCE","COLPAL","CAMS","CONCORDBIO","CONCOR",
        "COROMANDEL","CRAFTSMAN","CREDITACC","CROMPTON","CUMMINSIND","CYIENT","DCMSHRIRAM","DLF",
        "DOMS","DABUR","DALBHARAT","DATAPATTNS","DEEPAKFERT","DEEPAKNTR","DELHIVERY","DEVYANI",
        "DIVISLAB","DIXON","LALPATHLAB","DRREDDY","EIDPARRY","EIHOTEL","EICHERMOT","ELECON",
        "ELGIEQUIP","EMAMILTD","EMCURE","EMMVEE","ENDURANCE","ENGINERSIN","ERIS","ESCORTS",
        "ETERNAL","EXIDEIND","NYKAA","FEDERALBNK","FACT","FINCABLES","FSL","FIVESTAR",
        "FORCEMOT","FORTIS","GAIL","GMRAIRPORT","GABRIEL","GALLANTT","GRSE","GICRE",
        "GILLETTE","GLAND","GLAXO","GLENMARK","MEDANTA","GODIGIT","GPIL","GODFRYPHLP",
        "GODREJCP","GODREJIND","GODREJPROP","GRANULES","GRAPHITE","GRASIM","GRAVITA","GESHIP",
        "FLUOROCHEM","GMDCLTD","HEG","HBLENGINE","HCLTECH","HDBFS","HDFCAMC","HDFCBANK",
        "HDFCLIFE","HFCL","HAVELLS","HEROMOTOCO","HEXT","HSCL","HINDALCO","HAL",
        "HINDCOPPER","HINDPETRO","HINDUNILVR","HINDZINC","POWERINDIA","HOMEFIRST","HONASA","HONAUT",
        "HUDCO","HYUNDAI","ICICIBANK","ICICIGI","ICICIAMC","ICICIPRULI","IDBI","IDFCFIRSTB",
        "IFCI","IIFL","IRB","IRCON","ITCHOTELS","ITC","ITI","INDGN",
        "INDIACEM","INDIAMART","INDIANB","IEX","INDHOTEL","IOC","IOB","IRCTC",
        "IRFC","IREDA","IGL","INDUSTOWER","INDUSINDBK","NAUKRI","INFY","INOXWIND",
        "INTELLECT","INDIGO","IGIL","IKS","IPCALAB","JBCHEPHARM","JKCEMENT","JBMA",
        "JKTYRE","JMFINANCIL","JSWCEMENT","JSWDULUX","JSWENERGY","JSWINFRA","JSWSTEEL","JAINREC",
        "JPPOWER","JINDALSAW","JSL","JINDALSTEL","JIOFIN","JUBLFOOD","JUBLINGREA","JUBLPHARMA",
        "JWL","JYOTICNC","KPRMILL","KEI","KPITTECH","KAJARIACER","KPIL","KALYANKJIL",
        "KARURVYSYA","KAYNES","KEC","KFINTECH","KIRLOSENG","KOTAKBANK","KIMS","LTF",
        "LTTS","LGEINDIA","LICHSGFIN","LTFOODS","LTM","LT","LATENTVIEW","LAURUSLABS",
        "THELEELA","LEMONTREE","LENSKART","LICI","LINDEINDIA","LLOYDSME","LODHA","LUPIN",
        "MMTC","MRF","MGL","M&MFIN","M&M","MANAPPURAM","MRPL","MANKIND",
        "MARICO","MARUTI","MFSL","MAXHEALTH","MAZDOCK","MEESHO","MINDACORP","MSUMI",
        "MOTILALOFS","MPHASIS","MCX","MUTHOOTFIN","NATCOPHARM","NBCC","NCC","NHPC",
        "NLCINDIA","NMDC","NSLNISP","NTPCGREEN","NTPC","NH","NATIONALUM","NAVA",
        "NAVINFLUOR","NESTLEIND","NETWEB","NEULANDLAB","NEWGEN","NAM-INDIA","NIVABUPA","NUVAMA",
        "NUVOCO","OBEROIRLTY","ONGC","OIL","OLAELEC","OLECTRA","PAYTM","ONESOURCE",
        "OFSS","POLICYBZR","PCBL","PGEL","PIIND","PNBHOUSING","PTCIL","PVRINOX",
        "PAGEIND","PARADEEP","PATANJALI","PERSISTENT","PETRONET","PFIZER","PHOENIXLTD","PWL",
        "PIDILITIND","PINELABS","PIRAMALFIN","PPLPHARMA","POLYMED","POLYCAB","POONAWALLA","PFC",
        "POWERGRID","PREMIERENE","PRESTIGE","PNB","RRKABEL","RBLBANK","RECLTD","RHIM",
        "RITES","RADICO","RVNL","RAILTEL","RAINBOW","RKFORGE","REDINGTON","RELIANCE",
        "RPOWER","SBFC","SBICARD","SBILIFE","SJVN","SRF","SAGILITY","SAILIFE",
        "SAMMAANCAP","MOTHERSON","SAPPHIRE","SARDAEN","SAREGAMA","SCHAEFFLER","SCHNEIDER","SCI",
        "SHREECEM","SHRIRAMFIN","SHYAMMETL","ENRIN","SIEMENS","SIGNATURE","SOBHA","SOLARINDS",
        "SONACOMS","SONATSOFTW","STARHEALTH","SBIN","SAIL","SUMICHEM","SUNPHARMA","SUNTV",
        "SUNDARMFIN","SUPREMEIND","SPLPETRO","SUZLON","SWANCORP","SWIGGY","SYNGENE","SYRMA",
        "TBOTEK","TVSMOTOR","TATACAP","TATACHEM","TATACOMM","TCS","TATACONSUM","TATAELXSI",
        "TATAINVEST","TMCV","TMPV","TATAPOWER","TATASTEEL","TATATECH","TTML","TECHM",
        "TECHNOE","TEGA","TEJASNET","TENNIND","NIACL","RAMCOCEM","THERMAX","TIMKEN",
        "TITAGARH","TITAN","TORNTPHARM","TORNTPOWER","TARIL","TRAVELFOOD","TRENT","TRIDENT",
        "TRITURBINE","TIINDIA","UCOBANK","UNOMINDA","UPL","UTIAMC","ULTRACEMCO","UNIONBANK",
        "UBL","UNITDSPR","URBANCO","USHAMART","VTL","VBL","VEDL","VIJAYA"
    ]
    t = [s+".NS" for s in list(dict.fromkeys(symbols))]
    print(f"  Nifty 500    : {len(t)}"); return t

print("Loading tickers...")
sp500      = get_sp500()
r2000      = get_russell2000()
r2000_only = [t for t in r2000 if t not in set(sp500)]
nifty500   = get_nifty500() if INCLUDE_NIFTY else []
print(f"  Total        : {len(sp500)+len(r2000_only)+len(nifty500)}")

In [ ]:
# CELL 4 — Indicators
import pandas as pd
import numpy as np

def sma(s,n): return s.rolling(n).mean()
def ema(s,n): return s.ewm(span=n,adjust=False).mean()
def rsi(s,n=14):
    d=s.diff()
    g=d.clip(lower=0).rolling(n).mean()
    l=(-d.clip(upper=0)).rolling(n).mean()
    return 100-(100/(1+g/l.replace(0,np.nan)))
def _f(x):
    try: return float(x)
    except: return float("nan")
def _range(d,i):
    try: return _f(d["High"].iloc[-(i+1)]) - _f(d["Low"].iloc[-(i+1)])
    except: return float("nan")
def vwap_intraday(h):
    h=h.copy()
    h["_d"]=h.index.normalize()
    h["_tp"]=(h["High"]+h["Low"]+h["Close"])/3
    tpv=h.groupby("_d").apply(lambda g:(g["_tp"]*g["Volume"]).cumsum()).values
    cvol=h.groupby("_d")["Volume"].cumsum().values
    return pd.Series(tpv/cvol,index=h.index)
print("Indicators ready")

In [ ]:
# CELL 5 — Fetchers (daily/weekly + intraday separated)
import yfinance as yf
import pandas as pd
import time

def fetch_batch(tickers, period, interval, retries=2):
    """Daily / weekly batch fetcher."""
    out = {}
    if not tickers: return out
    for attempt in range(retries+1):
        try:
            raw = yf.download(tickers, period=period, interval=interval,
                              group_by="ticker", auto_adjust=False,
                              progress=False, threads=True)
            if raw.empty: break
            if len(tickers)==1:
                t=tickers[0]; df=raw.copy()
                if isinstance(df.columns,pd.MultiIndex): df.columns=df.columns.get_level_values(1)
                df=df.drop(columns=["Adj Close"],errors="ignore")
                df.dropna(how="all",inplace=True)
                if len(df)>5: out[t]=df
            else:
                for t in tickers:
                    try:
                        if t in raw.columns.levels[0]:
                            df=raw[t].copy()
                            if isinstance(df.columns,pd.MultiIndex): df.columns=df.columns.get_level_values(-1)
                            df=df.drop(columns=["Adj Close"],errors="ignore")
                            df=df.dropna(how="all")
                            if len(df)>5: out[t]=df
                    except: pass
            break
        except Exception as e:
            if any(x in str(e) for x in ["Rate","429","Too Many","RateLimit"]):
                wait=30*(attempt+1); print(f"  Rate limit — wait {wait}s"); time.sleep(wait)
            else: break
    return out

def fetch_intraday(tickers, period="5d", interval="1h", retries=2):
    """Intraday fetcher (1h) — uses working MultiIndex fix."""
    out = {}
    if not tickers: return out
    for attempt in range(retries+1):
        try:
            raw = yf.download(tickers, period=period, interval=interval,
                              group_by="ticker", auto_adjust=False,
                              progress=False, threads=True)
            if raw.empty: break
            if len(tickers)==1:
                t=tickers[0]; df=raw.copy()
                if isinstance(df.columns,pd.MultiIndex): df.columns=df.columns.get_level_values(1)
                df=df.drop(columns=["Adj Close"],errors="ignore")
                df.dropna(how="all",inplace=True)
                if len(df)>5: out[t]=df
            else:
                for t in tickers:
                    try:
                        if t in raw.columns.levels[0]:
                            df=raw[t].copy()
                            if isinstance(df.columns,pd.MultiIndex): df.columns=df.columns.get_level_values(-1)
                            df=df.drop(columns=["Adj Close"],errors="ignore")
                            df=df.dropna(how="all")
                            if len(df)>5: out[t]=df
                    except: pass
            break
        except Exception as e:
            if any(x in str(e) for x in ["Rate","429","Too Many","RateLimit"]):
                wait=30*(attempt+1); print(f"  Rate limit — wait {wait}s"); time.sleep(wait)
            else: break
    return out

print("fetch_batch (daily/weekly) + fetch_intraday (1h) ready")

In [ ]:
# CELL 6 — Daily scan functions
import numpy as np

def scan_bearish_momentum(d):
    try:
        if len(d)<52: return False
        c=_f(d["Close"].iloc[-1])
        return (c>50 and _f(rsi(d["Close"]).iloc[-1])<50
            and c<_f(d["Low"].iloc[-2])
            and c<_f(sma(d["Close"],50).iloc[-1])
            and _f(sma(d["Volume"],20).iloc[-1])>500000)
    except: return False

def scan_vcp(d, mktcap_m):
    try:
        if len(d)<252: return False
        c=_f(d["Close"].iloc[-1]); dvol=c*_f(sma(d["Volume"],20).iloc[-1])
        s200=_f(sma(d["Close"],200).iloc[-1]); s50=_f(sma(d["Close"],50).iloc[-1])
        c22=_f(d["Close"].iloc[-23]); c66=_f(d["Close"].iloc[-67])
        hi=_f(d["High"].rolling(252).max().iloc[-1])
        return ((c/c22>1.2 and mktcap_m>1 and dvol>30e6 and c>s200) or
                (c/c66>=1.3 and c>=1 and dvol>30e6 and c>s200) or
                (mktcap_m>=1000 and c>hi*0.75 and c>s50 and c>s200 and dvol>30e6))
    except: return False

def scan_satish_bullish(d, mktcap_m):
    try:
        if len(d)<47: return False
        c=_f(d["Close"].iloc[-1]); o=_f(d["Open"].iloc[-1])
        lo=_f(d["Low"].iloc[-1]); v=_f(d["Volume"].iloc[-1])
        pc=_f(d["Close"].iloc[-2]); pl=_f(d["Low"].iloc[-2]); ph=_f(d["High"].iloc[-2])
        pivot=(ph+pl+pc)/3; r=rsi(d["Close"])
        return (mktcap_m>=1000 and c>=100 and c>pivot
            and v>_f(sma(d["Volume"],10).iloc[-1])
            and lo>pl and c>pc and c>o
            and _f(sma(d["Close"],9).iloc[-1])>_f(ema(d["Close"],45).iloc[-1])
            and _f(r.iloc[-3])>_f(r.iloc[-2])
            and (c-pc)/pc*100>1 and _f(r.iloc[-1])>30 and v>100000)
    except: return False

def scan_hardik(d, w, mktcap_m, is_nifty=False):
    try:
        if len(d)<12 or len(w)<16: return False
        return (_f(rsi(w["Close"]).iloc[-1])>50
            and _f(d["Close"].iloc[-1])>_f(sma(d["Close"],10).iloc[-1])
            and _f(sma(d["Close"],10).iloc[-6])>_f(d["Close"].iloc[-6])
            and _f(d["Close"].iloc[-2])<_f(d["Open"].iloc[-2])
            and _f(d["Close"].iloc[-1])>_f(d["Open"].iloc[-1])
            and mktcap_m>5000)
    except: return False

def scan_nr6(d):
    try:
        if len(d)<8: return False
        r0=_range(d,0)
        return all(r0<_range(d,i) for i in range(1,7))
    except: return False

def scan_rocket_base(d):
    try:
        if len(d)<92: return False
        c=_f(d["Close"].iloc[-1])
        if c<=30 or _f(sma(d["Volume"],50).iloc[-1])<50000: return False
        return (c>=_f(d["Low"].iloc[-6])*1.2 or
                c>=_f(d["Low"].iloc[-31])*1.3 or
                c>=_f(d["Low"].iloc[-91])*1.3)
    except: return False

def scan_ep_breakout(d):
    try:
        if len(d)<130: return False
        max5=_f(d["Close"].rolling(5).max().iloc[-1])
        max120_6ago=_f(d["Close"].rolling(120).max().iloc[-7])
        return (max5>max120_6ago*1.05
            and _f(d["Volume"].iloc[-1])>_f(sma(d["Volume"],5).iloc[-1])
            and _f(d["Close"].iloc[-1])>_f(d["Close"].iloc[-2]))
    except: return False

def scan_20day_high(d, is_us=True):
    if not is_us: return False
    try:
        if len(d)<21: return False
        return _f(d["Close"].iloc[-1])>=_f(d["High"].rolling(20).max().iloc[-1])
    except: return False

def scan_2month_high(d, is_us=True):
    if not is_us: return False
    try:
        if len(d)<43: return False
        return _f(d["Close"].iloc[-1])>=_f(d["High"].rolling(42).max().iloc[-1])
    except: return False

def scan_gapup(d):
    try:
        if len(d)<2: return False
        return _f(d["Open"].iloc[-1])>=_f(d["Close"].iloc[-2])*1.03
    except: return False

def scan_up4pct(d):
    try:
        if len(d)<2: return False
        c=_f(d["Close"].iloc[-1]); o=_f(d["Open"].iloc[-1])
        return (c-o)/o>=0.04
    except: return False

print("Daily scan functions ready (11 scans)")

In [ ]:
# CELL 7 — Daily scan runner
import time

DAILY_SCAN_KEYS = [
    "bearish_momentum","vcp_setup","satish_bullish","hardik",
    "nr6","rocket_base","ep_breakout","high_20d","high_2month",
    "gapup_3pct","up_4pct"
]

def run_daily_scans(tickers, label):
    is_nifty = label=="Nifty 500"
    is_us    = not is_nifty
    hits = {k:[] for k in DAILY_SCAN_KEYS}
    batches=[tickers[i:i+BATCH_SIZE] for i in range(0,len(tickers),BATCH_SIZE)]
    total=len(batches)
    print(f"Scanning {label} — {len(tickers)} tickers, {total} batches")
    for n,batch in enumerate(batches):
        print(f"  Batch {n+1}/{total}", end="\r")
        daily  = fetch_batch(batch,"2y","1d")
        weekly = fetch_batch(batch,"5y","1wk")
        for t in batch:
            d=daily.get(t); w=weekly.get(t)
            if d is None or len(d)<10: continue
            try:
                c=float(d["Close"].iloc[-1]); av=float(sma(d["Volume"],20).iloc[-1])
                mktcap = c*av*30/(1e7 if is_nifty else 1e6)
            except: mktcap=0
            if scan_bearish_momentum(d): hits["bearish_momentum"].append(t)
            if scan_vcp(d,mktcap): hits["vcp_setup"].append(t)
            if scan_satish_bullish(d,mktcap): hits["satish_bullish"].append(t)
            if w is not None and scan_hardik(d,w,mktcap,is_nifty): hits["hardik"].append(t)
            if scan_nr6(d): hits["nr6"].append(t)
            if scan_rocket_base(d): hits["rocket_base"].append(t)
            if scan_ep_breakout(d): hits["ep_breakout"].append(t)
            if scan_20day_high(d,is_us): hits["high_20d"].append(t)
            if scan_2month_high(d,is_us): hits["high_2month"].append(t)
            if scan_gapup(d): hits["gapup_3pct"].append(t)
            if scan_up4pct(d): hits["up_4pct"].append(t)
        time.sleep(SLEEP_BETWEEN_BATCHES)
    print(f"\n{label} done")
    return hits

print("Daily runner ready")

In [ ]:
# CELL 8 — RUN DAILY SCANS
from datetime import datetime
start=datetime.now()
print(f"Started: {start.strftime('%H:%M:%S')}")
sp500_hits  = run_daily_scans(sp500,      "S&P 500")
r2000_hits  = run_daily_scans(r2000_only, "Russell 2000")
nifty_hits  = run_daily_scans(nifty500,   "Nifty 500") if INCLUDE_NIFTY else {k:[] for k in DAILY_SCAN_KEYS}
print(f"Done in ~{(datetime.now()-start).seconds//60} min")

# ── Save to Google Drive for Streamlit website ──────────────────
import os, pandas as pd
from datetime import datetime

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    _save_dir = '/content/drive/MyDrive/Stockbee'
    os.makedirs(_save_dir, exist_ok=True)
except Exception:
    _save_dir = '.'

_ts  = datetime.now().strftime('%Y-%m-%d %H:%M')
_rows = []
for _k in DAILY_SCAN_KEYS:
    _label = {"bearish_momentum":"Bearish Momentum","vcp_setup":"VCP Setup","satish_bullish":"Satish Bullish","hardik":"Hardik Scan","nr6":"NR6 Narrowest Range","rocket_base":"Rocket Base","ep_breakout":"EP Breakout","high_20d":"20-Day High (US)","high_2month":"2-Month High (US)","gapup_3pct":"Gap Up 3pct","up_4pct":"Up 4pct"}.get(_k, _k)
    for _t in sp500_hits.get(_k, []):  _rows.append({'Scan':_label,'Ticker':_t,'Universe':'S&P 500',  'Updated':_ts})
    for _t in r2000_hits.get(_k, []): _rows.append({'Scan':_label,'Ticker':_t,'Universe':'Russell 2000','Updated':_ts})
    for _t in nifty_hits.get(_k, []):  _rows.append({'Scan':_label,'Ticker':_t,'Universe':'Nifty 500', 'Updated':_ts})

_daily_df = pd.DataFrame(_rows) if _rows else pd.DataFrame(columns=['Scan','Ticker','Universe','Updated'])
_csv_path = os.path.join(_save_dir, 'daily_hits.csv')
_daily_df.to_csv(_csv_path, index=False)
print(f"\n✅ Saved daily_hits.csv → {_csv_path}  ({len(_daily_df)} rows)")
print(f"   Scans: {_daily_df['Scan'].nunique() if not _daily_df.empty else 0}  |  Tickers: {_daily_df['Ticker'].nunique() if not _daily_df.empty else 0}")

In [ ]:
# CELL 9 — Print summary
from datetime import datetime
LABELS={"bearish_momentum":"Bearish Momentum","vcp_setup":"VCP Setup",
    "satish_bullish":"Satish Bullish","hardik":"Hardik Scan",
    "nr6":"NR6 Narrowest Range","rocket_base":"Rocket Base",
    "ep_breakout":"EP Breakout","high_20d":"20-Day High (US)",
    "high_2month":"2-Month High (US)","gapup_3pct":"Gap Up 3pct","up_4pct":"Up 4pct"}
print("\n"+"="*65)
print(f"  DAILY RESULTS  {datetime.now().strftime('%d %b %Y  %H:%M')}")
print("="*65)
for k,label in LABELS.items():
    sp=sp500_hits.get(k,[]); r2k=r2000_hits.get(k,[]); nif=nifty_hits.get(k,[])
    tot=len(sp)+len(r2k)+len(nif)
    if tot==0: continue
    print(f"  {label:<38} S&P:{len(sp):>3} R2K:{len(r2k):>3} Nifty:{len(nif):>3} Total:{tot:>4}")
    if sp:  print(f"    S&P:   {chr(32).join(sp[:15])}")
    if r2k: print(f"    R2K:   {chr(32).join(r2k[:15])}")
    if nif: print(f"    Nifty: {chr(32).join(nif[:15])}")